Steps:
1. Import packages needed
2. Input data / Create Function for Lemmatizing Data
3. Creatures
<br>&emsp;3a. Prep data (includes lemmatize, train test split, normalize)
<br>&emsp;3b. Create model with Hyperparameter analysis
<br>&emsp;3c. Acquire parameters / Evaluate model
<br>&emsp;3d. Visualizations (GridSearch Heatmap, Scatterplot with best Gridsearch)
4. Planes Walkers
<br>&emsp;4a. Prep data (includes lemmatize, train test split, normalize)
<br>&emsp;4b. Create model with Hyperparameter analysis
<br>&emsp;4c. Acquire parameters / Evaluate model
<br>&emsp;4d. Visualizations (GridSearch Heatmap, Scatterplot with best Gridsearch)
5. Non-Creatures (Enchantments, Artifacts, and Instants)
<br>&emsp;5a. Prep data (includes lemmatize, train test split, normalize)
<br>&emsp;5b. Create model with Hyperparameter analysis
<br>&emsp;5c. Acquire parameters / Evaluate model
<br>&emsp;5d. Visualizations (GridSearch Heatmap, Scatterplot with best Gridsearch)

# 1. Import Packages

In [271]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [272]:
np.set_printoptions(precision=5)
random_state = 42
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.model_selection import validation_curve
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MultiLabelBinarizer

# 2. Input Data / Create Function for Lemmatizing Data

In [273]:
mtgjson_data_df = pd.read_csv('data/finalCards.csv')
mtgjson_data_df['text'] = mtgjson_data_df['text'].astype('string').fillna('')

data_before = [
    'isAlternative',
    'isGameChanger', 
    'isPromo',
    'isReprint', 
    'isReserved',
    'isHuman', 
    'isElemental',
    'isDragon', 
    'isSpirit', 
    'isAngel', 
    'isElf', 
    'isVampire', 
    'isZombie',
    'isBeast', 
    'isWizard', 
    'isSoldier', 
    'isKnight', 
    'isCleric', 
    'isWarrior',
    'isRogue', 
    'isShaman', 
    'isDruid', 
    'isCreature', 
    'isPlaneswalker', 
    'isMTGO', 
    'isFlying',
    'isLegal',
    'isBanned']

for col in data_before:
    mtgjson_data_df[col] = mtgjson_data_df[col].map({True: 1, False: 0})

ridge_lasso_data_df = mtgjson_data_df[['uuid', 
                            'cardName', 
                            'availability', 
                            'colorIdentity', 
                            'edhrecRank', 
                            'edhrecSaltiness', 
                            'finishes', 
                            'isAlternative', 
                            'isGameChanger',
                            'isPromo',
                            'isReprint',
                            'isReserved',
                            'manaValue',
                            'cardNumber',
                            'power',
                            'rarity',
                            'setCode',
                            'subtypes',
                            'text',
                            'toughness',
                            'types',
                            'price',
                            'commander',
                            'setName',
                            'releaseDate',
                            'scryfallId',
                            'isHuman', 
                            'isElemental',
                            'isDragon', 
                            'isSpirit', 
                            'isAngel', 
                            'isElf',
                            'isVampire', 
                            'isZombie',
                            'isBeast', 
                            'isWizard',
                            'isEldrazi', 
                            'isSoldier', 
                            'isKnight', 
                            'isCleric', 
                            'isWarrior',
                            'isRogue', 
                            'isShaman', 
                            'isDruid', 
                            'isCreature',
                            'isPlaneswalker',
                            'isMTGO', 
                            'isFlying',
                            'isLegal',
                            'isBanned'
                            ]]


ridge_lasso_data_df['finishes'] = ridge_lasso_data_df['finishes'].str.split(', ')
mlb = MultiLabelBinarizer()
encoded_finishes = mlb.fit_transform(ridge_lasso_data_df['finishes'])
finishes_df = pd.DataFrame(encoded_finishes, columns="finishes:_"+mlb.classes_)

ridge_lasso_data_df['colorIdentity'] = ridge_lasso_data_df['colorIdentity'].str.split(', ')
ci_mlb = MultiLabelBinarizer()
encoded_colorid = ci_mlb.fit_transform(ridge_lasso_data_df['colorIdentity'])
colorid_df = pd.DataFrame(encoded_colorid, columns="colorid:_"+ci_mlb.classes_)

rarity_df = pd.get_dummies(ridge_lasso_data_df['rarity'], prefix="rarity:")

ridge_lasso_data_df = pd.concat([ridge_lasso_data_df,rarity_df,finishes_df,colorid_df], axis=1)
ridge_lasso_data_df = ridge_lasso_data_df.drop(['rarity','finishes','colorIdentity'], axis=1)

column_uniqueval_df = {}
na_column_uniqueval = {}
for column in ridge_lasso_data_df.columns:
    if "id" in column or "Name" in column or "price" in column or 'Id' in column:
        ridge_lasso_data_df.dropna(subset=[column])
    elif "is" in column or "rarity" in column or "power" in column or "toughness" in column:
        ridge_lasso_data_df[column] = ridge_lasso_data_df[column].fillna(0)
    else:
        if pd.isna(ridge_lasso_data_df[column].unique().tolist()).any():
            na_column_uniqueval.update({column : ridge_lasso_data_df[column].unique()})
        else:
            column_uniqueval_df.update({column : ridge_lasso_data_df[column].unique()})

print(f'Quantity of cards PRIOR to dropping NA values: {len(ridge_lasso_data_df)}')
ridge_lasso_data_df = ridge_lasso_data_df.dropna()
print(f'Quantity of cards AFTER to dropping NA values: {len(ridge_lasso_data_df)}')

C:\Users\jaymj\AppData\Local\Temp\ipykernel_25212\1898253130.py:90: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ridge_lasso_data_df['finishes'] = ridge_lasso_data_df['finishes'].str.split(', ')
C:\Users\jaymj\AppData\Local\Temp\ipykernel_25212\1898253130.py:95: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ridge_lasso_data_df['colorIdentity'] = ridge_lasso_data_df['colorIdentity'].str.split(', ')


Quantity of cards PRIOR to dropping NA values: 94123
Quantity of cards AFTER to dropping NA values: 46113


### Add Lemmatization Step for cleaning text

In [274]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

# download libraries
nltk.download('wordnet')
nltk.download('punkt')

def lemmatize_sentence(sentence):
    lemmatizer = WordNetLemmatizer()
    #Should words be lemmatized? (Have to lemmatize words based on noun and verb...)
    tokens = word_tokenize(sentence)
    lemmatized_tokens = [lemmatizer.lemmatize(token)+' ' for token in tokens]
    return ''.join(lemmatized_tokens)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jaymj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jaymj\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


# 3. Creature Analysis

## 3a. Prep data (includes lemmatize, train test split, normalize)

#### Extract Creature Data and Lemmatize the ability text

In [275]:
creatures_df = ridge_lasso_data_df[ridge_lasso_data_df['isCreature'] == 1].copy()
creatures_df.head(10)

,uuid,cardName,availability,edhrecRank,edhrecSaltiness,isAlternative,isGameChanger,isPromo,isReprint,isReserved,...,finishes:_etched,finishes:_foil,finishes:_nonfoil,finishes:_signed,colorid:_B,colorid:_Colorless,colorid:_G,colorid:_R,colorid:_U,colorid:_W
0,00010d56-fe38-5e35-8aed-518019aa36a5,Sphinx of the Final Word,paper,10186.0,0.14,0,0,1,1,0,...,0,1,0,0,0,0,0,0,1,0
1,0001e0d0-2dcd-5640-aadc-a84765cf5fc9,Goblin King,paper,3454.0,0.34,0,0,0,1,0,...,0,0,1,0,0,0,0,1,0,0
3,0003d249-25d9-5223-af1e-1130f09622a7,Deadshot Minotaur,"mtgo, paper",25150.0,0.20,0,0,0,0,0,...,0,1,1,0,0,0,1,1,0,0
6,0005283f-d113-5937-ba52-a30570bfb334,Grave Titan,"mtgo, paper",1366.0,0.21,0,0,0,1,0,...,0,0,1,0,1,0,0,0,0,0
7,00054115-b2b6-5e22-a694-76fc8639eeb2,Merfolk Looter,paper,4221.0,0.24,0,0,0,1,0,...,0,0,1,1,0,0,0,0,1,0
8,0005d268-3fd0-5424-bc6b-573ecd713aa1,War Priest of Thune,"mtgo, paper",15342.0,0.18,0,0,0,1,0,...,0,1,1,0,0,0,0,0,0,1
10,0005fe8b-170d-5381-baaf-3d6bd58042f8,Soul Swindler,paper,14563.0,0.35,0,0,0,0,0,...,0,1,1,0,1,0,0,0,0,0
12,0009093e-de41-5432-9815-8d1f2efe16dd,Woe Strider,paper,1676.0,0.15,0,0,0,1,0,...,0,0,1,0,1,0,0,0,0,0
13,000a1b60-9edf-55c7-89e4-2e7bf96448d4,Corpse Appraiser,"arena, mtgo, paper",18424.0,0.26,0,0,0,0,0,...,0,1,1,0,1,0,0,1,1,0
15,000a85b2-d96c-5a74-81de-5d0a3a0357b3,"Chandra, Fire of Kaladesh // Chandra, Roaring ...","mtgo, paper",9214.0,0.38,0,0,0,1,0,...,0,1,0,0,0,0,0,1,0,0


In [276]:
#lemmatize creature ability text
creatures_df['text_lemmatized'] = creatures_df['text'].apply(lemmatize_sentence)

#### Vectorize text using TF-IDF and Count Quantities

In [277]:
#Acquire counts for each word (not including stop words)
creature_min_ngram = 1
creature_max_ngram = 3
stop_words = 'english'
vectorizer = 15

#TF IDF dataframe
creature_tfidf_vectorizer = TfidfVectorizer(max_features=vectorizer, stop_words=stop_words, ngram_range=(creature_min_ngram,creature_max_ngram))
creature_tfidf_model = creature_tfidf_vectorizer.fit_transform(creatures_df['text_lemmatized'])
creature_tfidf_ability_text = creature_tfidf_vectorizer.get_feature_names_out()
creature_tfidf_stop_words_vect = creature_tfidf_vectorizer.get_stop_words()
creature_tfidf_columns_text = ["contains_tfidf: "+ item for item in creature_tfidf_ability_text]
creature_tfidf_words_df = pd.DataFrame(creature_tfidf_model.toarray(), columns=creature_tfidf_columns_text)
creature_final_df = creatures_df.join(creature_tfidf_words_df).fillna(0)

#Count Vectorizer dataframe
creature_count_vectorizer = CountVectorizer(max_features=vectorizer, stop_words=stop_words, ngram_range=(creature_min_ngram,creature_max_ngram))
creature_count_model = creature_count_vectorizer.fit_transform(creatures_df['text_lemmatized'])
creature_count_ability_text = creature_count_vectorizer.get_feature_names_out()
creature_count_stop_words_vect = creature_count_vectorizer.get_stop_words()
creature_count_columns_text = ["contains_count: "+ item for item in creature_count_ability_text]
creature_count_words_df = pd.DataFrame(creature_count_model.toarray(), columns=creature_count_columns_text)
creature_final_df = creature_final_df.join(creature_count_words_df).fillna(0)

C:\Users\jaymj\AppData\Local\Temp\ipykernel_25212\1821162303.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  creature_final_df = creatures_df.join(creature_tfidf_words_df).fillna(0)


#### Train Test Split MTGJSON Data

In [278]:
creature_X_columns = [
        'power', 
        'toughness', 
        'price', 
        'edhrecRank',
        'edhrecSaltiness', 
        'manaValue',
        'isReprint',
        'isAlternative',
        'isGameChanger',
        'isPromo',
        'isReserved',
        'isHuman', 
        'isElemental',
        'isDragon', 
        'isSpirit', 
        'isAngel', 
        'isElf',
        'isVampire', 
        'isZombie',
        'isBeast', 
        'isWizard',
        'isEldrazi',
        'isSoldier', 
        'isKnight', 
        'isCleric', 
        'isWarrior',
        'isRogue', 
        'isShaman', 
        'isDruid', 
        'isMTGO', 
        'isFlying',
        'isLegal',
        'isBanned',
        'rarity:_common',
        'rarity:_mythic',
        'rarity:_rare',
        'rarity:_special',
        'rarity:_uncommon',
        'finishes:_etched',
        'finishes:_foil',
        'finishes:_nonfoil',
        'finishes:_signed',
        'colorid:_B',
        'colorid:_Colorless',
        'colorid:_G',
        'colorid:_R',
        'colorid:_U',
        'colorid:_W'
        ] + creature_tfidf_columns_text + creature_count_columns_text
creature_y_column = "price"
creature_X_columns.remove(creature_y_column)

creature_X = creature_final_df[creature_X_columns]
creature_y = creature_final_df[creature_y_column]

#Split Data
creature_X_train_raw, creature_X_test_raw, creature_y_train_raw, creature_y_test_raw = train_test_split(creature_X, creature_y, random_state=random_state)

#Normalize data for non binary columns
creature_scaled_columns = [
        'power',
        'toughness',
        'edhrecRank',
        'edhrecSaltiness',
        'manaValue']
creature_X_train_norm= creature_X_train_raw.copy()
creature_X_test_norm = creature_X_test_raw.copy()
creature_train_features = creature_X_train_norm[creature_scaled_columns]
creature_test_features = creature_X_test_norm[creature_scaled_columns]

creature_scaler = StandardScaler().fit(creature_train_features.values)
creature_train_features = creature_scaler.transform(creature_train_features.values)
creature_test_features = creature_scaler.transform(creature_test_features.values)

creature_X_train_norm[creature_scaled_columns] = creature_train_features
creature_X_test_norm[creature_scaled_columns] = creature_test_features

## 3b. Create Model with Hyperparameter Tuning

### Parameters to tune

In [279]:
#Parameters with and GridSearch for both Lasso and Ridge Regression
parameters = {
    'alpha' : [0.25, 0.5, 0.75, 1]
    }

### Ridge Model

In [315]:
import altair as alt
import math
alt.data_transformers.enable("vegafusion")
alt.renderers.enable("jupyter")

def ridge_model_with_graph(X_train, y_train, X_test, y_test, card_type, num_alphas = 200):

    n_alphas = 200
    alphas = np.logspace(-10.0, 10.0, n_alphas)
    coefs = []
    score = []
    best_model_score = -1.0
    best_model = None

    for a in alphas:
        ridge = Ridge(alpha=a, fit_intercept=False)
        ridge.fit(X_train, y_train)
        current_score = ridge.score(X_test, y_test)
        score.append({'Model_Reg_Lambda': a,
                      'Model_Score': current_score})
        coefs.append({'Coefficients':ridge.coef_,
                      'Features': X_train.columns.tolist(),
                     'Model_Reg_Lambda':a,
                      'Model_Score': current_score}
                      )
        if current_score > best_model_score:
            best_model = ridge
            best_model_score = current_score

    
    coefficients_data_df = pd.DataFrame(
        coefs,
        columns=['Coefficients', 'Features', 'Model_Reg_Lambda','Model_Score']
        ).explode(['Coefficients','Features'])
    

    coefficients_data_df.sort_values(
            by=['Model_Score','Model_Reg_Lambda', 'Coefficients'],
            ascending = [False, True, False],
            inplace=True
        )

    max_value = min(math.ceil(np.abs(coefficients_data_df['Coefficients']).max()*1.2),100)
    min_value = -max_value
    
    score_data_df = pd.DataFrame(
        score,
        columns=['Model_Reg_Lambda', 'Model_Score']
        )
    score_data_df.sort_values(
            by='Model_Score',
            inplace=True,
            ascending=False
            )
    
    top_20_scores = score_data_df['Model_Score'].tolist()[:21]
    features_score = pd.pivot_table(
        coefficients_data_df[coefficients_data_df['Model_Score'].isin(top_20_scores)],
        values= 'Coefficients',
        index='Features',
        aggfunc= 'mean').reset_index()
    features_score['Absolute_Coeff'] = np.abs(features_score['Coefficients'])
    features_top_20_score = features_score.sort_values(by='Absolute_Coeff', ascending=False)[:21]

    top_20_feature_data_df = coefficients_data_df[coefficients_data_df['Features'].isin(features_top_20_score['Features'])]

    line_plot = alt.Chart(top_20_feature_data_df).mark_line().encode(
        x=alt.X(
            "Model_Reg_Lambda:Q",
            title = "Model Regularization level (Lambda)",
            scale= alt.Scale(type = 'log')
            ),
        y=alt.Y(
            "Coefficients:Q",
            title = "Coefficient Weights",
            scale= alt.Scale(domain=[min_value,max_value])
        ),
        color=alt.Color(
            "Features:N",
            legend=alt.Legend(title="Features", titleFontSize=10, labelFontSize=8)
        ),
        tooltip=['Features:N']
    ).properties(
            width=800,
            height=450,
            title=alt.Title(text=f"Coefficient Path Plot for Top 20 {card_type} Card Elements", fontSize=28, anchor="middle"),
        ).configure_axis(
            labelFontSize=12,
            titleFontSize=22,
            gridOpacity=0.3).configure_view(strokeWidth=0).configure_legend(orient="right")
    
    
    
    return best_model, coefficients_data_df, score_data_df, line_plot


In [318]:
#Create Ridge Regression Model
creature_ridge_clf = Ridge()

#Fit model with TF-IDF
creature_X_train_norm_tfidf = creature_X_train_norm.copy().drop(columns=creature_count_columns_text)
creature_X_test_norm_tfidf = creature_X_test_norm.copy().drop(columns=creature_count_columns_text)
creature_ridge_clf_hyp = GridSearchCV(estimator=creature_ridge_clf, param_grid=parameters, cv=5)
creature_ridge_clf_hyp_tfidf = creature_ridge_clf_hyp.fit(creature_X_train_norm_tfidf, creature_y_train_raw)
creature_ridge_tfidf_score = creature_ridge_clf_hyp_tfidf.score(creature_X_test_norm_tfidf, creature_y_test_raw)
print("Ridge TF-IDF Score: ",creature_ridge_tfidf_score)

#Fit model with Count Quantities
creature_X_train_norm_count = creature_X_train_norm.copy().drop(columns=creature_tfidf_columns_text)
creature_X_test_norm_count =creature_X_test_norm.copy().drop(columns=creature_tfidf_columns_text)
creature_ridge_clf_hyp = GridSearchCV(estimator=creature_ridge_clf, param_grid=parameters, cv=5)
creature_ridge_clf_hyp_count = creature_ridge_clf_hyp.fit(creature_X_train_norm_count, creature_y_train_raw)
creature_ridge_count_score = creature_ridge_clf_hyp_count.score(creature_X_test_norm_count, creature_y_test_raw)
print("Ridge Count Score: ",creature_ridge_count_score)


#Identify best model
if creature_ridge_tfidf_score > creature_ridge_count_score:
    creature_X_train_norm_ridge = creature_X_train_norm.copy().drop(columns=creature_count_columns_text)
    creature_X_test_norm_ridge = creature_X_test_norm.copy().drop(columns=creature_count_columns_text)
    if creature_X_train_norm_ridge.columns.str.contains("contains_count: ").any():
        print("CHECK COLUMNS")
        print(creature_X_train_norm_ridge.columns[creature_X_train_norm_ridge.columns.str.contains("contains_count: ")].tolist())
    else:
        print("COLUMNS ARE GOOD")
        print("USE TDF-IDF SCORE FOR RIDGE MODEL")
else:
    creature_X_train_norm_ridge = creature_X_train_norm.copy().drop(columns=creature_tfidf_columns_text)
    creature_X_test_norm_ridge = creature_X_test_norm.copy().drop(columns=creature_tfidf_columns_text)
    if creature_X_train_norm_ridge.columns.str.contains("contains_tfidf: ", case=False).any():
        print("CHECK COLUMNS")
        print(creature_X_train_norm_ridge.columns[creature_X_train_norm_ridge.columns.str.contains("contains_tfidf: ")].tolist())
    else:
        print("COLUMNS ARE GOOD")
        print("USE COUNT QUANTITIES FOR LASSO MODEL")

creature_ridge_clf_hyp, creat_r_coeffs_df, creat_r_score_df, creat_r_line_plot = ridge_model_with_graph(
    creature_X_train_norm_ridge, 
    creature_y_train_raw, 
    creature_X_test_norm_ridge,
    creature_y_test_raw,
    "Creature",
    num_alphas=2000)

Ridge TF-IDF Score:  0.0052688517684987834
Ridge Count Score:  0.005134712987157175
COLUMNS ARE GOOD
USE TDF-IDF SCORE FOR RIDGE MODEL


## 3c. Acquire Parameters / Evaluate Metrics

In [319]:
creature_ridge_parameters = creature_ridge_clf_hyp.get_params
creature_ridge_y_predict = creature_ridge_clf_hyp.predict(creature_X_test_norm_ridge)
creature_ridge_mse = mean_squared_error(creature_y_test_raw, creature_ridge_y_predict)
creature_ridge_best_score = creat_r_score_df.iloc[0,1]
creature_ridge_alpha_value = creat_r_score_df.iloc[0,0]

print("RIDGE MODEL for CREATURES")
print(f"Best Regulation value (Lambda/Alpha Score): {creature_ridge_alpha_value}")
print(f"Best R-Square Score of Test Data: {creature_ridge_best_score}")
print(f"Mean Square error: {creature_ridge_mse}")
print("----------------------------------------")

RIDGE MODEL for CREATURES
Best Regulation value (Lambda/Alpha Score): 28.66067616948256
Best R-Square Score of Test Data: 0.005421155891756624
Mean Square error: 48519.28470348126
----------------------------------------


In [351]:
#Evaluate feature_importance for ridge Model
creature_ridge_feature_importance = pd.DataFrame(creat_r_coeffs_df[creat_r_coeffs_df['Model_Score'].isin([creature_ridge_best_score])])

creature_ridge_feature_importance['Abs_Val_Coefficients'] = abs(creature_ridge_feature_importance['Coefficients'])
creature_ridge_feature_importance.sort_values('Abs_Val_Coefficients', ascending=False, inplace=True)
print("RIDGE MODEL")
print(creature_ridge_feature_importance.head(25))
print("----------------------------------------")

RIDGE MODEL
    Coefficients                   Features  Model_Reg_Lambda  Model_Score  \
114    68.655195                 isReserved         28.660676     0.005421   
114    41.663582             rarity:_mythic         28.660676     0.005421   
114    34.158474            rarity:_special         28.660676     0.005421   
114     33.16686               rarity:_rare         28.660676     0.005421   
114    28.967559           rarity:_uncommon         28.660676     0.005421   
114    27.781722             rarity:_common         28.660676     0.005421   
114    -19.70398           finishes:_signed         28.660676     0.005421   
114   -16.117428           finishes:_etched         28.660676     0.005421   
114   -14.971262                    isPromo         28.660676     0.005421   
114    11.707431              isAlternative         28.660676     0.005421   
114   -10.183853          finishes:_nonfoil         28.660676     0.005421   
114    -9.272574                     isMTGO         

## 3d. Visualizations

#### Residual Plot Function

In [348]:
import altair as alt
alt.data_transformers.enable("vegafusion")
alt.renderers.enable("jupyter")

def residual_plot(y_raw, y_pred, title="Residual Plot"):

    residuals = y_raw - y_pred
    std_residual = np.std(residuals)

    residual_data_df = pd.DataFrame(
        {'Residuals': residuals,
        'Fitted_Values' : y_pred,
            }
    )

    residual_data_df = residual_data_df[(np.abs(residual_data_df['Residuals']) < 3 * std_residual)]

    residual_data_df['isOutlier'] = np.where(np.abs(residual_data_df['Residuals']) > 2 * std_residual,"Outlier", "Normal")


    scatter_plot = alt.Chart(residual_data_df).mark_point().encode(
        x=alt.X("Fitted_Values:Q", title = "Predicted Price Values"),
        y=alt.Y(
            "Residuals:Q",
            title = "Residual Error between Predicted and Actual Price Value",
            scale=alt.Scale(domain=[-3*std_residual, 3*std_residual])
        ),
        color=alt.Color(
            "isOutlier:N",
            scale=alt.Scale(domain=["Normal", "Outlier"], range=["#009E73", "#D55E00"]),
            legend=alt.Legend(title="Normality", titleFontSize=18, labelFontSize=16)
        ),
        tooltip=['Fitted_Values:Q', 'Residuals:Q']
    )

    zero_line = alt.Chart(pd.DataFrame({'y':[0]})).mark_rule(
        color="#333333",
        strokeWidth=2,
        strokeDash=[6,3],
        ).encode(
        y=alt.Y('y:Q')
    )

    std_lines = alt.Chart(
        pd.DataFrame(
            {'y':[2*std_residual,-2*std_residual],
                'labels': ['2 std deviation above mean',
                        '-2*std deviation below mean'
                    ]        
                }
            )
        ).mark_rule(
        color="#999999",
        strokeWidth=2,
        strokeDash=[6,3],
        ).encode(
        y=alt.Y('y:Q')
    )

    chart = (
        alt.layer(zero_line, std_lines, scatter_plot)
        .properties(
            width=800,
            height=800,
            title=alt.Title(text=title, fontSize=28, anchor="middle"),
        )
        .configure_axis(labelFontSize=12, titleFontSize=22, gridOpacity=0.3)
        .configure_view(strokeWidth=0)
        .configure_legend(orient="right")
    )

    return chart

In [349]:
creature_chart = residual_plot(creature_y_test_raw, creature_ridge_y_predict, "Residual Plot for MTG Creature Cards")
creature_chart.show()

#### Coefficient Path Plot Function

In [323]:
creat_r_line_plot.show()

# 4. Planeswalker Analysis

## 4a. Prep data (includes lemmatize, train test split, normalize)

#### Extract Creature Data and Lemmatize the ability text

In [324]:
planeswalk_df = ridge_lasso_data_df[(ridge_lasso_data_df['isPlaneswalker'] == 1)]
planeswalk_df.head(10)

,uuid,cardName,availability,edhrecRank,edhrecSaltiness,isAlternative,isGameChanger,isPromo,isReprint,isReserved,...,finishes:_etched,finishes:_foil,finishes:_nonfoil,finishes:_signed,colorid:_B,colorid:_Colorless,colorid:_G,colorid:_R,colorid:_U,colorid:_W
43,001e28fe-5bc1-57ef-9814-adc1195a297b,"Liliana, the Last Hope",paper,7063.0,0.26,1,0,0,1,0,...,0,1,0,0,1,0,0,0,0,0
85,00396ed6-b91d-5c78-8760-bb22d8f3e9ca,"Tevesh Szat, Doom of Fools","mtgo, paper",3174.0,0.57,0,0,0,0,0,...,0,1,1,0,1,0,0,0,0,0
87,003d9ecd-53bc-55c7-993f-26ec405f71e4,"Gideon, Martial Paragon","mtgo, paper",15747.0,0.37,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1
219,009c0efc-d03f-5a9d-8df9-d0fd2e94b4ca,"Liliana, Waker of the Dead",paper,5185.0,0.79,1,0,1,1,0,...,0,1,0,0,1,0,0,0,0,0
281,00be1adb-2ca4-5485-ba27-94f0e96eb2d9,"Minsc & Boo, Timeless Heroes",paper,3343.0,0.54,1,0,1,1,0,...,0,1,0,0,0,0,1,1,0,0
312,00d36b0f-e866-54df-b4aa-447ba0f681e2,"Huatli, the Sun's Heart","arena, mtgo, paper",7026.0,0.53,0,0,0,0,0,...,0,1,1,0,0,0,1,0,0,1
323,00d9bc14-e2dd-5385-bdca-560cc3b8a4be,"Kasmina, Enigmatic Mentor",paper,6160.0,0.48,0,0,1,1,0,...,0,1,0,0,0,0,0,0,1,0
482,014cb967-e80c-5373-b937-fe6aa41cc452,"Ob Nixilis, the Adversary",paper,7084.0,0.13,1,0,1,1,0,...,0,1,0,0,1,0,0,1,0,0
549,017d5ecf-098a-54b8-a6a6-58e3d9b2c307,"Ugin, the Spirit Dragon",paper,1419.0,1.17,0,0,1,1,0,...,0,1,1,0,0,1,0,0,0,0
570,018e64c0-8949-5ce2-82ea-20792d4158c7,"Tezzeret, Betrayer of Flesh",paper,3178.0,0.30,1,0,1,1,0,...,0,1,0,0,0,0,0,0,1,0


In [325]:
#lemmatize planeswalker ability text
planeswalk_df['text_lemmatized'] = planeswalk_df['text'].apply(lemmatize_sentence)

C:\Users\jaymj\AppData\Local\Temp\ipykernel_25212\3748780994.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  planeswalk_df['text_lemmatized'] = planeswalk_df['text'].apply(lemmatize_sentence)


#### Vectorize text using TF-IDF and Count Quantities

In [326]:
#Acquire counts for each word (not including stop words)
planeswalk_min_ngram = 1
planeswalk_max_ngram = 3
stop_words = 'english'

#TF IDF dataframe
planeswalk_tfidf_vectorizer = TfidfVectorizer(max_features=1, stop_words=stop_words, ngram_range=(planeswalk_min_ngram,planeswalk_max_ngram))
planeswalk_tfidf_model = planeswalk_tfidf_vectorizer.fit_transform(planeswalk_df['text_lemmatized'])
planeswalk_tfidf_ability_text = planeswalk_tfidf_vectorizer.get_feature_names_out()
planeswalk_tfidf_stop_words_vect = planeswalk_tfidf_vectorizer.get_stop_words()
planeswalk_tfidf_columns_text = ["contains_tfidf: "+ item for item in planeswalk_tfidf_ability_text]
planeswalk_tfidf_words_df = pd.DataFrame(planeswalk_tfidf_model.toarray(), columns=planeswalk_tfidf_columns_text)
planeswalk_final_df = planeswalk_df.join(planeswalk_tfidf_words_df).fillna(0)

#Count Vectorizer dataframe
planeswalk_count_vectorizer = CountVectorizer(max_features=8, stop_words=stop_words, ngram_range=(planeswalk_min_ngram,planeswalk_max_ngram))
planeswalk_count_model = planeswalk_count_vectorizer.fit_transform(planeswalk_df['text_lemmatized'])
planeswalk_count_ability_text = planeswalk_count_vectorizer.get_feature_names_out()
planeswalk_count_stop_words_vect = planeswalk_count_vectorizer.get_stop_words()
planeswalk_count_columns_text = ["contains_count: "+ item for item in planeswalk_count_ability_text]
planeswalk_count_words_df = pd.DataFrame(planeswalk_count_model.toarray(), columns=planeswalk_count_columns_text)
planeswalk_final_df = planeswalk_final_df.join(planeswalk_count_words_df).fillna(0)

C:\Users\jaymj\AppData\Local\Temp\ipykernel_25212\2698409365.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  planeswalk_final_df = planeswalk_df.join(planeswalk_tfidf_words_df).fillna(0)


#### Train Test Split MTGJSON Data

In [327]:
planeswalk_X_columns = [
        'power', 
        'toughness', 
        'price', 
        'edhrecRank',
        'edhrecSaltiness', 
        'manaValue',
        'isReprint',
        'isAlternative',
        'isGameChanger',
        'isPromo',
        'isReserved',
        'isHuman', 
        'isElemental',
        'isDragon', 
        'isSpirit', 
        'isAngel', 
        'isElf',
        'isVampire', 
        'isZombie',
        'isBeast', 
        'isWizard',
        'isEldrazi',
        'isSoldier', 
        'isKnight', 
        'isCleric', 
        'isWarrior',
        'isRogue', 
        'isShaman', 
        'isDruid', 
        'isCreature', 
        'isMTGO', 
        'isFlying',
        'isLegal',
        'isBanned',
        'rarity:_common',
        'rarity:_mythic',
        'rarity:_rare',
        'rarity:_special',
        'rarity:_uncommon',
        'finishes:_etched',
        'finishes:_foil',
        'finishes:_nonfoil',
        'finishes:_signed',
        'colorid:_B',
        'colorid:_Colorless',
        'colorid:_G',
        'colorid:_R',
        'colorid:_U',
        'colorid:_W'
        ] + planeswalk_tfidf_columns_text + planeswalk_count_columns_text
planeswalk_y_column = "price"
planeswalk_X_columns.remove(planeswalk_y_column)

planeswalk_X = planeswalk_final_df[planeswalk_X_columns]
planeswalk_y = planeswalk_final_df[planeswalk_y_column]

#Split Data
planeswalk_X_train_raw, planeswalk_X_test_raw, planeswalk_y_train_raw, planeswalk_y_test_raw = train_test_split(planeswalk_X, planeswalk_y, random_state=random_state)

#Normalize data for non binary columns
planeswalk_scaled_columns = [
        'power',
        'toughness',
        'edhrecRank',
        'edhrecSaltiness',
        'manaValue']
planeswalk_X_train_norm= planeswalk_X_train_raw.copy()
planeswalk_X_test_norm = planeswalk_X_test_raw.copy()
planeswalk_train_features = planeswalk_X_train_norm[planeswalk_scaled_columns]
planeswalk_test_features = planeswalk_X_test_norm[planeswalk_scaled_columns]

planeswalk_scaler = StandardScaler().fit(planeswalk_train_features.values)
planeswalk_train_features = planeswalk_scaler.transform(planeswalk_train_features.values)
planeswalk_test_features = planeswalk_scaler.transform(planeswalk_test_features.values)

planeswalk_X_train_norm[planeswalk_scaled_columns] = planeswalk_train_features
planeswalk_X_test_norm[planeswalk_scaled_columns] = planeswalk_test_features


## 4b. Create Model with Hyperparameter Tuning

### Parameters to tune

In [328]:
# Utilize same parameters as Creature Analysis

### Ridge Model

In [329]:
#Create Ridge Regression Model
planeswalk_ridge_clf = Ridge()

#Fit model with TF-IDF
planeswalk_X_train_norm_tfidf = planeswalk_X_train_norm.copy().drop(columns=planeswalk_count_columns_text)
planeswalk_X_test_norm_tfidf = planeswalk_X_test_norm.copy().drop(columns=planeswalk_count_columns_text)
planeswalk_ridge_clf_hyp = GridSearchCV(estimator=planeswalk_ridge_clf, param_grid=parameters, cv=5)
planeswalk_ridge_clf_hyp_tfidf = planeswalk_ridge_clf_hyp.fit(planeswalk_X_train_norm_tfidf, planeswalk_y_train_raw)
planeswalk_ridge_tfidf_score = planeswalk_ridge_clf_hyp_tfidf.score(planeswalk_X_test_norm_tfidf, planeswalk_y_test_raw)
print("Ridge TF-IDF Score: ",planeswalk_ridge_tfidf_score)

#Fit model with Count Quantities
planeswalk_X_train_norm_count = planeswalk_X_train_norm.copy().drop(columns=planeswalk_tfidf_columns_text)
planeswalk_X_test_norm_count =planeswalk_X_test_norm.copy().drop(columns=planeswalk_tfidf_columns_text)
planeswalk_ridge_clf_hyp = GridSearchCV(estimator=planeswalk_ridge_clf, param_grid=parameters, cv=5)
planeswalk_ridge_clf_hyp_count = planeswalk_ridge_clf_hyp.fit(planeswalk_X_train_norm_count, planeswalk_y_train_raw)
planeswalk_ridge_count_score = planeswalk_ridge_clf_hyp_count.score(planeswalk_X_test_norm_count, planeswalk_y_test_raw)
print("Ridge Count Score: ",planeswalk_ridge_count_score)


#Identify best model
if planeswalk_ridge_tfidf_score > planeswalk_ridge_count_score:
    planeswalk_X_train_norm_ridge = planeswalk_X_train_norm.copy().drop(columns=planeswalk_count_columns_text)
    planeswalk_X_test_norm_ridge = planeswalk_X_test_norm.copy().drop(columns=planeswalk_count_columns_text)
    if planeswalk_X_train_norm_ridge.columns.str.contains("contains_count: ").any():
        print("CHECK COLUMNS")
        print(planeswalk_X_train_norm_ridge.columns[planeswalk_X_train_norm_ridge.columns.str.contains("contains_count: ")].tolist())
    else:
        print("COLUMNS ARE GOOD")
        print("USE TDF-IDF SCORE FOR RIDGE MODEL")
else:
    planeswalk_X_train_norm_ridge = planeswalk_X_train_norm.copy().drop(columns=planeswalk_tfidf_columns_text)
    planeswalk_X_test_norm_ridge = planeswalk_X_test_norm.copy().drop(columns=planeswalk_tfidf_columns_text)
    if planeswalk_X_train_norm_ridge.columns.str.contains("contains_tfidf: ", case=False).any():
        print("CHECK COLUMNS")
        print(planeswalk_X_train_norm_ridge.columns[planeswalk_X_train_norm_ridge.columns.str.contains("contains_tfidf: ")].tolist())
    else:
        print("COLUMNS ARE GOOD")
        print("USE COUNT QUANTITIES FOR LASSO MODEL")

planeswalk_ridge_clf_hyp, pw_r_coeffs_df, pw_r_score_df, pw_r_line_plot = ridge_model_with_graph(
    planeswalk_X_train_norm_ridge, 
    planeswalk_y_train_raw, 
    planeswalk_X_test_norm_ridge,
    planeswalk_y_test_raw,
    "Planeswalker",
    num_alphas=2000)

Ridge TF-IDF Score:  0.06887125178944875
Ridge Count Score:  0.06345228764069477
COLUMNS ARE GOOD
USE TDF-IDF SCORE FOR RIDGE MODEL


## 4c. Acquire Parameters / Evaluate Metrics

In [330]:
planeswalk_ridge_parameters = planeswalk_ridge_clf_hyp.get_params
planeswalk_ridge_y_predict = planeswalk_ridge_clf_hyp.predict(planeswalk_X_test_norm_ridge)
planeswalk_ridge_mse = mean_squared_error(planeswalk_y_test_raw, planeswalk_ridge_y_predict)
planeswalk_ridge_best_score = pw_r_score_df.iloc[0,1]
planeswalk_ridge_alpha_value = pw_r_score_df.iloc[0,0]

print("RIDGE MODEL for PLANESWALKERS")
print(f"Best Regulation value (Lambda/Alpha Score): {planeswalk_ridge_alpha_value}")
print(f"Best R-Square Score of Test Data: {planeswalk_ridge_best_score}")
print(f"Mean Square error: {planeswalk_ridge_mse}")
print("----------------------------------------")

RIDGE MODEL for PLANESWALKERS
Best Regulation value (Lambda/Alpha Score): 1e-10
Best R-Square Score of Test Data: 0.06938412719049825
Mean Square error: 3056.327260026644
----------------------------------------


In [352]:
#Evaluate feature_importance for ridge Model
planeswalk_ridge_feature_importance = pd.DataFrame(pw_r_coeffs_df[pw_r_coeffs_df['Model_Score'].isin([planeswalk_ridge_best_score])])


planeswalk_ridge_feature_importance['Abs_Val_Coefficients'] = abs(planeswalk_ridge_feature_importance['Coefficients'])
planeswalk_ridge_feature_importance.sort_values('Abs_Val_Coefficients', ascending=False, inplace=True)
print("RIDGE MODEL")
print(planeswalk_ridge_feature_importance.head(25))
print("----------------------------------------")

RIDGE MODEL
  Coefficients                  Features  Model_Reg_Lambda  Model_Score  \
0   -19.181037             isGameChanger      1.000000e-10     0.069384   
0     19.11828                   isLegal      1.000000e-10     0.069384   
0   -15.560643              rarity:_rare      1.000000e-10     0.069384   
0   -12.387293          rarity:_uncommon      1.000000e-10     0.069384   
0    10.688977            finishes:_foil      1.000000e-10     0.069384   
0    -8.139271                    isMTGO      1.000000e-10     0.069384   
0    -6.349297         finishes:_nonfoil      1.000000e-10     0.069384   
0      -5.9327  contains_tfidf: creature      1.000000e-10     0.069384   
0    -5.604133                edhrecRank      1.000000e-10     0.069384   
0    -5.224242             isAlternative      1.000000e-10     0.069384   
0    -4.843672                colorid:_B      1.000000e-10     0.069384   
0    -4.299635                colorid:_W      1.000000e-10     0.069384   
0     4.27661

## 4d. Visualizations

#### Residual Plot Function

In [332]:
planeswalk_chart = residual_plot(planeswalk_y_test_raw, planeswalk_ridge_y_predict, "Residual Plot for MTG Planeswalker Cards")
planeswalk_chart.show()

#### Coefficient Path Plot Function

In [333]:
pw_r_line_plot.show()

# 5. Non-Creature Analysis

## 5a. Prep data (includes lemmatize, train test split, normalize)

#### Extract Non-Creature Data and Lemmatize the ability text

In [334]:
non_creature_df = ridge_lasso_data_df[(ridge_lasso_data_df['isCreature'] == 0) & (ridge_lasso_data_df['isPlaneswalker'] == 0)]
print(len(non_creature_df))
non_creature_df.head(10)

6853


,uuid,cardName,availability,edhrecRank,edhrecSaltiness,isAlternative,isGameChanger,isPromo,isReprint,isReserved,...,finishes:_etched,finishes:_foil,finishes:_nonfoil,finishes:_signed,colorid:_B,colorid:_Colorless,colorid:_G,colorid:_R,colorid:_U,colorid:_W
20,000db0e0-047e-59e2-b7aa-265e656ea91c,Entangler,"mtgo, paper",10095.0,0.14,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,1
22,000ed184-57ec-5b1e-a900-5d770a945999,Infinite Reflection,"mtgo, paper",16521.0,0.33,0,0,0,0,0,...,0,1,1,0,0,0,0,0,1,0
49,002219b5-cd5e-561e-a752-aa2986e31ecd,Mechtitan Core,paper,4017.0,0.15,1,0,1,1,0,...,0,1,0,0,0,1,0,0,0,0
100,004b3b84-db1d-5191-91d9-0b5844b8cf11,Peppersmoke,"mtgo, paper",15134.0,0.18,0,0,0,1,0,...,0,1,1,0,1,0,0,0,0,0
103,004d21aa-69bc-508b-8a76-3fc5975ce6ab,Lance,paper,26974.0,0.20,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,1
107,004ea5e7-bd25-5fcc-a4bb-c7f619068867,Ward of Piety,"mtgo, paper",28587.0,0.40,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,1
114,00516945-5600-539d-90c2-8568a6b18790,Murmuring Bosk,paper,2504.0,0.13,0,0,0,1,0,...,0,0,1,0,1,0,1,0,0,1
120,0056d472-f526-5728-8104-67e1c6915666,Perception Bobblehead,"mtgo, paper",5724.0,0.06,0,0,0,0,0,...,0,1,1,0,0,1,0,0,0,0
137,005ff425-d8d7-550f-8ade-7d2a545031da,Entangling Vines,"mtgo, paper",27071.0,0.05,0,0,0,0,0,...,0,1,1,0,0,0,1,0,0,0
138,00605bc0-86d5-58bb-8007-1ef050832405,Weatherlight Compleated,"arena, mtgo, paper",8457.0,0.29,0,0,0,0,0,...,0,1,1,0,0,1,0,0,0,0


In [335]:
#lemmatize non-creature ability text
non_creature_df['text_lemmatized'] = non_creature_df['text'].apply(lemmatize_sentence)

C:\Users\jaymj\AppData\Local\Temp\ipykernel_25212\678399806.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  non_creature_df['text_lemmatized'] = non_creature_df['text'].apply(lemmatize_sentence)


#### Vectorize text using TF-IDF and Count Quantities

In [336]:
#Acquire counts for each word (not including stop words)
non_creature_min_ngram = 1
non_creature_max_ngram = 3
stop_words = 'english'

#TF IDF dataframe
non_creature_tfidf_vectorizer = TfidfVectorizer(max_features=17, stop_words=stop_words, ngram_range=(non_creature_min_ngram,non_creature_max_ngram))
non_creature_tfidf_model = non_creature_tfidf_vectorizer.fit_transform(non_creature_df['text_lemmatized'])
non_creature_tfidf_ability_text = non_creature_tfidf_vectorizer.get_feature_names_out()
non_creature_tfidf_stop_words_vect = non_creature_tfidf_vectorizer.get_stop_words()
non_creature_tfidf_columns_text = ["contains_tfidf: "+ item for item in non_creature_tfidf_ability_text]
non_creature_tfidf_words_df = pd.DataFrame(non_creature_tfidf_model.toarray(), columns=non_creature_tfidf_columns_text)
non_creature_final_df = non_creature_df.join(non_creature_tfidf_words_df).fillna(0)

#Count Vectorizer dataframe
non_creature_count_vectorizer = CountVectorizer(max_features=11, stop_words=stop_words, ngram_range=(non_creature_min_ngram,non_creature_max_ngram))
non_creature_count_model = non_creature_count_vectorizer.fit_transform(non_creature_df['text_lemmatized'])
non_creature_count_ability_text = non_creature_count_vectorizer.get_feature_names_out()
non_creature_count_stop_words_vect = non_creature_count_vectorizer.get_stop_words()
non_creature_count_columns_text = ["contains_count: "+ item for item in non_creature_count_ability_text]
non_creature_count_words_df = pd.DataFrame(non_creature_count_model.toarray(), columns=non_creature_count_columns_text)
non_creature_final_df = non_creature_final_df.join(non_creature_count_words_df).fillna(0)

C:\Users\jaymj\AppData\Local\Temp\ipykernel_25212\3118133033.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  non_creature_final_df = non_creature_df.join(non_creature_tfidf_words_df).fillna(0)


#### Train Test Split MTGJSON Data

In [337]:
non_creature_X_columns = [
        'power', 
        'toughness', 
        'price', 
        'edhrecRank',
        'edhrecSaltiness', 
        'manaValue',
        'isReprint',
        'isAlternative',
        'isGameChanger',
        'isPromo',
        'isReserved',
        'isHuman', 
        'isElemental',
        'isDragon', 
        'isSpirit', 
        'isAngel', 
        'isElf',
        'isVampire', 
        'isZombie',
        'isBeast', 
        'isWizard',
        'isEldrazi',
        'isSoldier', 
        'isKnight', 
        'isCleric', 
        'isWarrior',
        'isRogue', 
        'isShaman', 
        'isDruid', 
        'isCreature',
        'isPlaneswalker',
        'isMTGO', 
        'isFlying',
        'isLegal',
        'isBanned',
        'rarity:_common',
        'rarity:_mythic',
        'rarity:_rare',
        'rarity:_special',
        'rarity:_uncommon',
        'finishes:_etched',
        'finishes:_foil',
        'finishes:_nonfoil',
        'finishes:_signed',
        'colorid:_B',
        'colorid:_Colorless',
        'colorid:_G',
        'colorid:_R',
        'colorid:_U',
        'colorid:_W'
        ] + non_creature_tfidf_columns_text + non_creature_count_columns_text
non_creature_y_column = "price"
non_creature_X_columns.remove(non_creature_y_column)

non_creature_X = non_creature_final_df[non_creature_X_columns]
non_creature_y = non_creature_final_df[non_creature_y_column]

#Split Data
non_creature_X_train_raw, non_creature_X_test_raw, non_creature_y_train_raw, non_creature_y_test_raw = train_test_split(non_creature_X, non_creature_y, random_state=random_state)

#Normalize data for non binary columns
non_creature_scaled_columns = [
        'power',
        'toughness',
        'edhrecRank',
        'edhrecSaltiness',
        'manaValue']
non_creature_X_train_norm= non_creature_X_train_raw.copy()
non_creature_X_test_norm = non_creature_X_test_raw.copy()
non_creature_train_features = non_creature_X_train_norm[non_creature_scaled_columns]
non_creature_test_features = non_creature_X_test_norm[non_creature_scaled_columns]

non_creature_scaler = StandardScaler().fit(non_creature_train_features.values)
non_creature_train_features = non_creature_scaler.transform(non_creature_train_features.values)
non_creature_test_features = non_creature_scaler.transform(non_creature_test_features.values)

non_creature_X_train_norm[non_creature_scaled_columns] = non_creature_train_features
non_creature_X_test_norm[non_creature_scaled_columns] = non_creature_test_features


## 5b. Create Model with Hyperparameter Tuning

### Parameters to tune

In [338]:
# Utilize same parameters as Creature Analysis

### Ridge Model

In [339]:
#Create Ridge Regression Model
non_creature_ridge_clf = Ridge()

#Fit model with TF-IDF
non_creature_X_train_norm_tfidf = non_creature_X_train_norm.copy().drop(columns=non_creature_count_columns_text)
non_creature_X_test_norm_tfidf = non_creature_X_test_norm.copy().drop(columns=non_creature_count_columns_text)
non_creature_ridge_clf_hyp = GridSearchCV(estimator=non_creature_ridge_clf, param_grid=parameters, cv=5)
non_creature_ridge_clf_hyp_tfidf = non_creature_ridge_clf_hyp.fit(non_creature_X_train_norm_tfidf, non_creature_y_train_raw)
non_creature_ridge_tfidf_score = non_creature_ridge_clf_hyp_tfidf.score(non_creature_X_test_norm_tfidf, non_creature_y_test_raw)
print("Ridge TF-IDF Score: ",non_creature_ridge_tfidf_score)

#Fit model with Count Quantities
non_creature_X_train_norm_count = non_creature_X_train_norm.copy().drop(columns=non_creature_tfidf_columns_text)
non_creature_X_test_norm_count =non_creature_X_test_norm.copy().drop(columns=non_creature_tfidf_columns_text)
non_creature_ridge_clf_hyp = GridSearchCV(estimator=non_creature_ridge_clf, param_grid=parameters, cv=5)
non_creature_ridge_clf_hyp_count = non_creature_ridge_clf_hyp.fit(non_creature_X_train_norm_count, non_creature_y_train_raw)
non_creature_ridge_count_score = non_creature_ridge_clf_hyp_count.score(non_creature_X_test_norm_count, non_creature_y_test_raw)
print("Ridge Count Score: ",non_creature_ridge_count_score)


#Identify best model
if non_creature_ridge_tfidf_score > non_creature_ridge_count_score:
    non_creature_X_train_norm_ridge = non_creature_X_train_norm.copy().drop(columns=non_creature_count_columns_text)
    non_creature_X_test_norm_ridge = non_creature_X_test_norm.copy().drop(columns=non_creature_count_columns_text)
    if non_creature_X_train_norm_ridge.columns.str.contains("contains_count: ").any():
        print("CHECK COLUMNS")
        print(non_creature_X_train_norm_ridge.columns[non_creature_X_train_norm_ridge.columns.str.contains("contains_count: ")].tolist())
    else:
        print("COLUMNS ARE GOOD")
        print("USE TDF-IDF SCORE FOR RIDGE MODEL")
else:
    non_creature_X_train_norm_ridge = non_creature_X_train_norm.copy().drop(columns=non_creature_tfidf_columns_text)
    non_creature_X_test_norm_ridge = non_creature_X_test_norm.copy().drop(columns=non_creature_tfidf_columns_text)
    if non_creature_X_train_norm_ridge.columns.str.contains("contains_tfidf: ", case=False).any():
        print("CHECK COLUMNS")
        print(non_creature_X_train_norm_ridge.columns[non_creature_X_train_norm_ridge.columns.str.contains("contains_tfidf: ")].tolist())
    else:
        print("COLUMNS ARE GOOD")
        print("USE COUNT QUANTITIES FOR LASSO MODEL")

non_creature_ridge_clf_hyp, non_creat_r_coeffs_df, non_creat_r_score_df, non_creat_r_line_plot = ridge_model_with_graph(
    non_creature_X_train_norm_ridge, 
    non_creature_y_train_raw, 
    non_creature_X_test_norm_ridge,
    non_creature_y_test_raw,
    "Non-Creature",
    num_alphas=200)

Ridge TF-IDF Score:  0.0016662889191764174
Ridge Count Score:  0.0016709624874883255
COLUMNS ARE GOOD
USE COUNT QUANTITIES FOR LASSO MODEL


## 5c. Acquire Parameters / Evaluate Metrics

In [340]:
non_creature_ridge_parameters = non_creature_ridge_clf_hyp.get_params
non_creature_ridge_y_predict = non_creature_ridge_clf_hyp.predict(non_creature_X_test_norm_ridge)
non_creature_ridge_mse = mean_squared_error(non_creature_y_test_raw, non_creature_ridge_y_predict)
non_creature_ridge_best_score = non_creat_r_score_df.iloc[0,1]
non_creature_ridge_alpha_value = non_creat_r_score_df.iloc[0,0]

print("RIDGE MODEL for NON-CREATURES")
print(f"Best Regulation value (Lambda/Alpha Score): {non_creature_ridge_alpha_value}")
print(f"Best R-Square Score of Test Data: {non_creature_ridge_best_score}")
print(f"Mean Square error: {non_creature_ridge_mse}")
print("----------------------------------------")

RIDGE MODEL for NON-CREATURES
Best Regulation value (Lambda/Alpha Score): 91.15888299750837
Best R-Square Score of Test Data: 0.0045435528166372086
Mean Square error: 3901273.9701414793
----------------------------------------


In [353]:
#Evaluate feature_importance for ridge Model
non_creature_ridge_feature_importance = pd.DataFrame(non_creat_r_coeffs_df[non_creat_r_coeffs_df['Model_Score'].isin([non_creature_ridge_best_score])])

non_creature_ridge_feature_importance['Abs_Val_Coefficients'] = abs(non_creature_ridge_feature_importance['Coefficients'])
non_creature_ridge_feature_importance.sort_values('Abs_Val_Coefficients', ascending=False, inplace=True)
print("RIDGE MODEL")
print(non_creature_ridge_feature_importance.head(25))
print("----------------------------------------")

RIDGE MODEL
    Coefficients                    Features  Model_Reg_Lambda  Model_Score  \
119   393.294056                  isReserved         91.158883     0.004544   
119    61.963313                     isLegal         91.158883     0.004544   
119   -40.969263                      isMTGO         91.158883     0.004544   
119    33.839398             edhrecSaltiness         91.158883     0.004544   
119    32.128376                  colorid:_U         91.158883     0.004544   
119   -29.965702                   isReprint         91.158883     0.004544   
119    25.773752                  colorid:_R         91.158883     0.004544   
119   -24.572951                     isPromo         91.158883     0.004544   
119    23.834027                  colorid:_B         91.158883     0.004544   
119   -23.120222              finishes:_foil         91.158883     0.004544   
119   -22.012916                   manaValue         91.158883     0.004544   
119   -17.167968            finishes:_et

## 5d. Visualizations

#### Residual Plot

In [342]:
non_creature_chart = residual_plot(non_creature_y_test_raw, non_creature_ridge_y_predict, "Residual Plot for MTG Non-Creature Cards")
non_creature_chart.show()

#### Coefficient Path Plot Function

In [343]:
non_creat_r_line_plot.show()